# Road Damage Detection — INT422 Project
**Course:** INT422 (Deep Learning)  |  **Topic:** Road Damage Detection

This notebook covers:
1. Environment setup
2. Dataset download (RDD2022 - India subset recommended)
3. Convert Pascal VOC XML labels → YOLO format
4. Train YOLOv8 model
5. Evaluate (mAP, precision, recall, confusion matrix)
6. Inference on new images/video
7. Export model for deployment (web app)

> Run cells top to bottom. Use Colab GPU: Runtime → Change runtime type → GPU (T4).

## 1. Setup

In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless
import ultralytics
ultralytics.checks()

## 2. Get the dataset

**Option A (recommended — fastest):** Use a pre-converted YOLO-format RDD dataset from Roboflow Universe (search "Road Damage Detection RDD2022 YOLO"). Get a free API key from roboflow.com, then:
```python
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT_NAME")
dataset = project.version(1).download("yolov8")
```

**Option B (raw RDD2022, needs conversion):** Clone the official repo and use the XML→YOLO converter below.

In [ ]:
# OPTION B: download official RDD2022 (India subset shown as example URL structure)
# Full dataset info: https://github.com/sekilab/RoadDamageDetector
# Uncomment and set the correct country zip URL from the repo's README before running.

# !wget -q <PASTE_COUNTRY_ZIP_URL_HERE> -O rdd_india.zip
# !unzip -q rdd_india.zip -d /content/rdd_raw

In [ ]:
# XML (Pascal VOC) -> YOLO txt converter
import os, glob, xml.etree.ElementTree as ET

CLASSES = ["D00", "D10", "D20", "D40"]  # longitudinal, transverse, alligator crack, pothole

def convert_voc_to_yolo(xml_dir, img_dir, out_label_dir):
    os.makedirs(out_label_dir, exist_ok=True)
    for xml_file in glob.glob(os.path.join(xml_dir, "*.xml")):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find("size")
        w, h = int(size.find("width").text), int(size.find("height").text)
        lines = []
        for obj in root.findall("object"):
            cls = obj.find("name").text
            if cls not in CLASSES:
                continue
            cls_id = CLASSES.index(cls)
            b = obj.find("bndbox")
            xmin, ymin = float(b.find("xmin").text), float(b.find("ymin").text)
            xmax, ymax = float(b.find("xmax").text), float(b.find("ymax").text)
            xc = ((xmin + xmax) / 2) / w
            yc = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        name = os.path.splitext(os.path.basename(xml_file))[0]
        with open(os.path.join(out_label_dir, name + ".txt"), "w") as f:
            f.write("\n".join(lines))
    print("Conversion done.")

# Example usage once raw data is downloaded:
# convert_voc_to_yolo("/content/rdd_raw/India/train/annotations/xmls",
#                      "/content/rdd_raw/India/train/images",
#                      "/content/dataset/labels/train")

## 3. Dataset YAML

In [ ]:
yaml_content = """
path: /content/dataset
train: images/train
val: images/val

names:
  0: D00_longitudinal_crack
  1: D10_transverse_crack
  2: D20_alligator_crack
  3: D40_pothole
"""
with open("/content/dataset/data.yaml", "w") as f:
    f.write(yaml_content)
print(yaml_content)

## 4. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano - fast, good for Colab free GPU. Use yolov8s.pt for better accuracy.

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="road_damage",
    name="yolov8n_rdd",
    patience=15
)

## 5. Evaluate

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 6. Inference (single image / batch)

In [ ]:
best_model = YOLO("road_damage/yolov8n_rdd/weights/best.pt")

results = best_model.predict(source="/content/dataset/images/val", conf=0.35, save=True)

# Simple severity scoring: bigger bbox area relative to image = higher severity
for r in results[:3]:
    img_area = r.orig_shape[0] * r.orig_shape[1]
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        area_pct = ((x2 - x1) * (y2 - y1)) / img_area * 100
        cls_name = r.names[int(box.cls[0])]
        severity = "High" if area_pct > 8 else "Medium" if area_pct > 3 else "Low"
        print(f"{cls_name} | conf={float(box.conf[0]):.2f} | area%={area_pct:.1f} | severity={severity}")

## 7. Export for deployment
Download `best.pt` and use it in the Flask web app (see accompanying `webapp/` files).

In [ ]:
from google.colab import files
files.download("road_damage/yolov8n_rdd/weights/best.pt")